# Project: The "TechStore" Data Platform

## Part 1: Data Extraction

### Source 1: The ERP System (MySql)

In [3]:
# note for members remeber to install mysql-connector-python using pip install
import mysql.connector
import pandas as pd

#create a connection to the database
db = mysql.connector.connect(
    host="boughida.com",
    user="student_user_4ing",
    password="bi_guelma_2025",
    database="techstore_erp"
)

# create a cursor object to interact with the database, and fetch all table names
mycursor = db.cursor()
mycursor.execute("SELECT table_name FROM information_schema.tables WHERE table_schema = 'techstore_erp'")
tables_names = mycursor.fetchall()

# dictionary that will hold dataframes for each table, i used a dictionary instead of direct variable bcz in case when the database is updated (added new table or removed one) the code will still work without any modification
dfs = {}

# iterate over each table name, fetch its data, and store it in a dataframe
for table_name in tables_names:
    mycursor.execute(f"SELECT * FROM {table_name[0]}")
    
    # fetch all rows from the table
    rows = mycursor.fetchall()
    
    # get column names
    cols = [col[0] for col in mycursor.description]
    
    # create a dataframe and store it in the dictionary
    dfs[table_name[0]] = pd.DataFrame(rows, columns=cols)
    
print(dfs.keys())

dict_keys(['table_stores', 'table_customers', 'table_subcategories', 'table_sales', 'table_reviews', 'table_products', 'table_cities', 'table_categories'])


### Source 2: Departmental Files (Excel files)

In [4]:
df_marketing_expenses = pd.read_excel("Excel Files/marketing_expenses.xlsx")
df_monthly_targets = pd.read_excel("Excel Files/monthly_targets.xlsx")
df_shipping_rates = pd.read_excel("Excel Files/shipping_rates.xlsx")

print(df_marketing_expenses.head())
print(df_monthly_targets.head())
print(df_shipping_rates.head())

                  Date     Category Campaign_Type  Marketing_Cost_USD
0  2023-01-01 00:00:00    Computers  Social Media               900.0
1  2023-01-01 00:00:00  Smartphones  Social Media              2123.0
2  2023-01-01 00:00:00        Audio  Social Media               449.0
3  2023-01-01 00:00:00      Cameras            TV               563.0
4  2023-01-01 00:00:00     Printers            TV              1992.0
  Store_ID                Month Target_Revenue    Manager_Name
0       S1  2023-01-01 00:00:00        5429539  Billel Rahmani
1        2  2023-01-01 00:00:00        7808052  Khadidja Talbi
2        3  2023-01-01 00:00:00        4066978    Ryad Guellil
3        4  2023-01-01 00:00:00        3483297      Walid Diaf
4  Store_5  2023-01-01 00:00:00        3478480     Ryad Benali
  region_name    provider  shipping_cost  average_delivery_days
0       South    Yalidine           1005                      7
1        West   Kazi Tour            499                      2
2      Cen

### Source 3: Competitor Pricing (Web Scraping)

In [5]:
# note for members remeber to install requests and beautifulsoup4 using pip install
import requests
from bs4 import BeautifulSoup

# list hold all the website's pages to scrape
pages = ["https://boughida.com/competitor/index.html", "https://boughida.com/competitor/competitor_page_2.html", "https://boughida.com/competitor/competitor_page_3.html"]

rows = []

for URL in pages:
    # retrieve and parse the HTML data for each page
    page = requests.get(URL)
    soup = BeautifulSoup(page.content, "html.parser")
    product_cards = soup.find_all("div", class_="product-card")
    
    for product_card in product_cards:
        # extract product name and price
        product_name = product_card.find(
            "h5", class_="card-title text-primary product-name"
        ).get_text(strip=True)
        product_price = product_card.find(
            "span", class_="product-price"
        ).get_text(strip=True)
        
        # append the extracted data to the rows list
        rows.append({
            "product_name": product_name,
            "product_price": product_price
        })

# store the scraped data in a dataframe
df_product_prices_competitor = pd.DataFrame(rows)
print(df_product_prices_competitor)

                  product_name product_price
0            Samsung S23 Ultra    174800 DZD
1              HP LaserJet Pro     46000 DZD
2   Best Deal: Dell 24 Monitor     24200 DZD
3               Canon i-Sensys     39000 DZD
4                Epson EcoTank     27700 DZD
5           Promo: Nikon D3500     69200 DZD
6         Xiaomi Redmi Note 12     36800 DZD
7          HP Pavilion Desktop     83000 DZD
8                 Redmi Buds 4      5300 DZD
9                   DJI Mini 3    126900 DZD
10        Promo: Sony SRS-XB13     11100 DZD
11              ASUS ROG STRIX    290600 DZD
12                LG UltraGear     41700 DZD
13                IPHONE CABLE      3200 DZD
14             Lenovo ThinkPad     94600 DZD
15                 AirPods Pro     47400 DZD
16          Phone Case Silicon      1800 DZD
17             Sony WH-1000XM5     65400 DZD
18                   iPhone 13    136900 DZD
19             Canon 445 Black      2600 DZD
20                 DJI Mavic 3    350400 DZD
21        

### Source 4: Legacy Archives (OCR)

In [4]:
# note for members remeber to install pytesseract using pip install
from PIL import Image, ImageOps
import pytesseract
import re

images = ["Invoices/order_001.jpg", "Invoices/order_002.jpg", "Invoices/order_003.jpg", "Invoices/order_004.jpg", "Invoices/order_005.jpg"]

rows = []

for img in images:
    # open each image
    image = Image.open(img)
    
    # scale it to grey to improve OCR accuracy (bcz the thinner the text the harder to read it is for OCR)
    gray_image = ImageOps.grayscale(image)
    
    # resizing the dimensions of the image to make text more clear for OCR
    scale_factor = 2
    resized_image = gray_image.resize((gray_image.width * scale_factor, gray_image.height * scale_factor), resample=Image.LANCZOS)
    
    # extract text from the resized image using pytesseract with French language setting
    output = pytesseract.image_to_string(resized_image, lang="fra")

    # using regex library to extract the desired fields from the output
    # date field
    date = re.search(r"Date:\s*([\d-]+)", output)
    date = date.group(1) if date else "Not found"

    # costumerID field
    customer_id = re.search(r"Client ID:\s*(\w+)", output)
    customer_id = customer_id.group(1) if customer_id else "Not found"

    # splitting the output into lines
    lines = [line.strip() for line in output.splitlines() if line.strip()]
    
    # extracting the product details line
    product_line = re.compile(r"^(.+?)\s+(\d+)\s+(\d+)\s+(\d+)$") # regex pattern to match product details line
    for line in lines:
        matched = product_line.match(line) # try to match the current line with the product pattern
        if matched: # if the line matches the pattern, then extract the details
            product_name = matched.group(1).strip()
            quantity = int(matched.group(2))
            total_revenue = int(matched.group(4))
            break
        else:
            # if the current line doesn't match the product details pattern, set default values (to indicate that the extraction failed for this one)
            product_name = quantity = total_revenue = "Not found"

    rows.append({
        "Date": date,
        "CustomerID": customer_id,
        "Product Name": product_name,
        "Quantity": quantity,
        "Total Revenue": total_revenue
    })
    
    
df_invoices = pd.DataFrame(rows)
print(df_invoices)

         Date CustomerID       Product Name  Quantity  Total Revenue
0  2022-09-22      C1001       HP Victus 15         2         250000
1  2022-01-20      C1003     MacBook Air M2         1         195000
2  2022-06-01      C1004  Samsung S23 Ultra         3         555000
3  2022-10-27      C1025      iPhone 14 Pro         3         690000
4  2022-02-27      C1002        Dell XPS 13         2         520000


## Part 2: Data processing

#### Cleaning ERP tables

In [10]:
import os
import numpy as np
from sklearn.impute import SimpleImputer


print("="*60)
print("CLEANING ERP TABLES (MYSQL DATA)")
print("="*60)

# Create directories for cleaned data
erp_cleaned_dir = "Cleaned ERP Data"
os.makedirs(erp_cleaned_dir, exist_ok=True)

# Check if we have the ERP data
if 'dfs' not in locals() or not isinstance(dfs, dict) or len(dfs) == 0:
    print(" ERROR: 'dfs' dictionary not found or empty!")
    print("   Make sure you ran the MySQL extraction code first.")
    print("   The code that starts with: import mysql.connector")
    dfs = {}  # Create empty dict to avoid errors
else:
    print(f"✓ Found {len(dfs)} ERP tables")
    print(f"  Tables: {list(dfs.keys())}")

# Initialize counters
successful_tables = 0
failed_tables = 0

# Clean each ERP table
for table_name, df_original in dfs.items():
    print(f"\n{'='*50}")
    print(f"CLEANING TABLE: {table_name}")
    print(f"{'='*50}")
    
    try:
        # Create a copy to avoid modifying original
        df_clean = df_original.copy()
        
        # 1. Display original info
        print(f"Original shape: {df_clean.shape}")
        print(f"Columns ({len(df_clean.columns)}): {list(df_clean.columns)}")
        
        # 2. Remove duplicates
        initial_rows = len(df_clean)
        duplicates = df_clean.duplicated().sum()
        if duplicates > 0:
            df_clean = df_clean.drop_duplicates()
            print(f"Removed {duplicates} duplicate(s)")
        else:
            print("No duplicates found")
        
        # 3. Check and handle missing values
        missing_total = df_clean.isnull().sum().sum()
        if missing_total > 0:
            print(f"\nHandling {missing_total} missing values...")
            
            for col in df_clean.columns:
                missing_count = df_clean[col].isnull().sum()
                if missing_count > 0:
                    print(f"  {col}: {missing_count} missing")
                    
                    # Handle based on column type and name
                    if pd.api.types.is_numeric_dtype(df_clean[col]):
                        # For numeric columns
                        if missing_count == len(df_clean):
                            # Entire column is missing - fill with 0
                            df_clean[col] = 0
                            print(f"    Entire column was empty, filled with 0")
                        else:
                            # Use SimpleImputer for partial missing values
                            try:
                                imputer = SimpleImputer(strategy='median')
                                imputed_data = imputer.fit_transform(df_clean[[col]])
                                df_clean[col] = imputed_data.flatten()
                                print(f"    Filled with median: {imputer.statistics_[0]:.2f}")
                            except:
                                # Fallback: fill with column median
                                median_val = df_clean[col].median(skipna=True)
                                df_clean[col] = df_clean[col].fillna(median_val if not pd.isna(median_val) else 0)
                                print(f"    Filled with median (fallback): {median_val:.2f}")
                    else:
                        # For text/object columns
                        if any(keyword in col.lower() for keyword in ['id', 'code']):
                            df_clean[col] = df_clean[col].fillna('UNKNOWN')
                            print(f"    Filled with 'UNKNOWN'")
                        elif any(keyword in col.lower() for keyword in ['name', 'title', 'category']):
                            df_clean[col] = df_clean[col].fillna('Unknown')
                            print(f"    Filled with 'Unknown'")
                        else:
                            df_clean[col] = df_clean[col].fillna('')
                            print(f"    Filled with empty string")
        else:
            print("No missing values found")
        
        # 4. Clean text columns
        text_columns = df_clean.select_dtypes(include=['object']).columns
        if len(text_columns) > 0:
            print(f"\nCleaning {len(text_columns)} text column(s)...")
            
            for col in text_columns:
                # Convert to string and strip whitespace
                df_clean[col] = df_clean[col].astype(str).str.strip()
                
                # Skip if it's an ID/code column (keep as-is)
                if not any(keyword in col.lower() for keyword in ['id', 'code', 'key']):
                    # Capitalize first letter of each word for name/title columns
                    if any(keyword in col.lower() for keyword in ['name', 'title', 'category', 'type', 'description']):
                        df_clean[col] = df_clean[col].str.title()
                        print(f"  {col}: Applied title case")
        
        # 5. Handle negative values in numeric columns
        numeric_columns = df_clean.select_dtypes(include=[np.number]).columns
        negative_counts = {}
        
        for col in numeric_columns:
            negative_mask = df_clean[col] < 0
            negative_count = negative_mask.sum()
            if negative_count > 0:
                negative_counts[col] = negative_count
        
        if negative_counts:
            print(f"\nHandling negative values:")
            for col, count in negative_counts.items():
                print(f"  {col}: {count} negative value(s)")
                
                # Check if negative values make sense for this column
                if any(keyword in col.lower() for keyword in ['cost', 'price', 'revenue', 'amount', 'quantity']):
                    # For financial/numeric columns, convert to absolute value
                    df_clean.loc[df_clean[col] < 0, col] = df_clean.loc[df_clean[col] < 0, col].abs()
                    print(f"    Converted to positive (absolute value)")
                else:
                    # For other columns (like IDs), negative values don't make sense
                    # Replace with 0 or appropriate value
                    df_clean.loc[df_clean[col] < 0, col] = 0
                    print(f"    Replaced with 0 (invalid negative)")
        
        # 6. Convert appropriate numeric columns to integers
        print(f"\nConverting numeric columns to appropriate types...")
        for col in numeric_columns:
            if col.lower() not in ['price', 'cost', 'rate', 'percentage', 'ratio']:
                # Convert to int if all values are whole numbers
                if df_clean[col].notna().all() and (df_clean[col] % 1 == 0).all():
                    df_clean[col] = df_clean[col].astype(int)
                    print(f"  {col}: Converted to integer")
        
        # 7. Save the cleaned table
        print(f"\nSaving cleaned table...")
        
        # Choose format based on size
        if len(df_clean) > 1000000:  # More than 1 million rows
            file_path = os.path.join(erp_cleaned_dir, f"{table_name}_cleaned.csv")
            df_clean.to_csv(file_path, index=False)
            print(f"  ✓ Saved as CSV (large table): {file_path}")
        else:
            file_path = os.path.join(erp_cleaned_dir, f"{table_name}_cleaned.xlsx")
            df_clean.to_excel(file_path, index=False, engine='openpyxl')
            print(f"  ✓ Saved as Excel: {file_path}")
        
        # Verify the saved file
        if os.path.exists(file_path):
            file_size = os.path.getsize(file_path)
            file_size_mb = file_size / (1024 * 1024)
            print(f"  File size: {file_size_mb:.2f} MB")
            
            # Try to read it back for verification
            try:
                if file_path.endswith('.csv'):
                    test_read = pd.read_csv(file_path, nrows=5)
                else:
                    test_read = pd.read_excel(file_path, nrows=5, engine='openpyxl')
                
                print(f"  ✓ File can be read back (tested first 5 rows)")
                successful_tables += 1
                
                # Store in dictionary for later use
                if 'erp_cleaned_dfs' not in locals():
                    erp_cleaned_dfs = {}
                erp_cleaned_dfs[table_name] = df_clean
                
            except Exception as e:
                print(f"  ✗ Warning: Could not read back file: {str(e)}")
                failed_tables += 1
        else:
            print(f"  ✗ Error: File was not created!")
            failed_tables += 1
        
        # Display summary for this table
        print(f"\n{table_name} Cleaning Summary:")
        print(f"  Original rows: {initial_rows}")
        print(f"  After cleaning: {len(df_clean)}")
        print(f"  Removed duplicates: {duplicates}")
        print(f"  Missing values handled: {missing_total}")
        
        # Display sample of cleaned data
        print(f"\nSample of cleaned data (first 3 rows):")
        print(df_clean.head(3))
        
    except Exception as e:
        print(f"\n ERROR cleaning {table_name}:")
        print(f"   Error type: {type(e).__name__}")
        print(f"   Error message: {str(e)}")
        failed_tables += 1
        
        # Try to save original as backup
        try:
            backup_path = os.path.join(erp_cleaned_dir, f"{table_name}_original_backup.xlsx")
            df_original.to_excel(backup_path, index=False, engine='openpyxl')
            print(f"   Saved original as backup: {backup_path}")
        except:
            print(f"   Could not save backup either")

print("\n" + "="*60)
print("ERP TABLES CLEANING COMPLETE - SUMMARY")
print("="*60)

if successful_tables + failed_tables > 0:
    print(f"\n RESULTS:")
    print(f"  Successful: {successful_tables} table(s)")
    print(f"  Failed: {failed_tables} table(s)")
    print(f"  Total processed: {successful_tables + failed_tables} table(s)")
    
    if successful_tables > 0:
        print(f"\n SUCCESSFULLY CLEANED TABLES:")
        cleaned_files = []
        if os.path.exists(erp_cleaned_dir):
            cleaned_files = [f for f in os.listdir(erp_cleaned_dir) 
                           if f.endswith(('.xlsx', '.csv')) and '_cleaned' in f]
            
            for file in sorted(cleaned_files):
                file_path = os.path.join(erp_cleaned_dir, file)
                try:
                    if file.endswith('.csv'):
                        df_temp = pd.read_csv(file_path, nrows=1)
                    else:
                        df_temp = pd.read_excel(file_path, nrows=1, engine='openpyxl')
                    print(f"  ✓ {file}: {df_temp.shape[1]} columns")
                except:
                    print(f"  ✗ {file}: Could not verify")
    
    if failed_tables > 0:
        print(f"\n FAILED TABLES:")
        print(f"  {failed_tables} table(s) could not be cleaned properly")
        print(f"  Check error messages above for details")
        
    print(f"\n OUTPUT DIRECTORY:")
    print(f"  {os.path.abspath(erp_cleaned_dir)}")
    
    # List all files in the directory
    if os.path.exists(erp_cleaned_dir):
        all_files = os.listdir(erp_cleaned_dir)
        if all_files:
            print(f"  Files in directory ({len(all_files)} total):")
            for file in sorted(all_files):
                file_path = os.path.join(erp_cleaned_dir, file)
                if os.path.isfile(file_path):
                    size = os.path.getsize(file_path)
                    size_kb = size / 1024
                    print(f"    {file} ({size_kb:.1f} KB)")
else:
    print("\n No tables were processed.")
    print("  Make sure you have extracted data from MySQL first.")
    print("  Check that 'dfs' dictionary contains your ERP tables.")


CLEANING ERP TABLES (MYSQL DATA)
✓ Found 8 ERP tables
  Tables: ['table_stores', 'table_customers', 'table_subcategories', 'table_sales', 'table_reviews', 'table_products', 'table_cities', 'table_categories']

CLEANING TABLE: table_stores
Original shape: (12, 3)
Columns (3): ['Store_ID', 'Store_Name', 'City_ID']
No duplicates found
No missing values found

Cleaning 1 text column(s)...
  Store_Name: Applied title case

Converting numeric columns to appropriate types...
  Store_ID: Converted to integer
  City_ID: Converted to integer

Saving cleaned table...
  ✓ Saved as Excel: Cleaned ERP Data\table_stores_cleaned.xlsx
  File size: 0.01 MB
  ✓ File can be read back (tested first 5 rows)

table_stores Cleaning Summary:
  Original rows: 12
  After cleaning: 12
  Removed duplicates: 0
  Missing values handled: 0

Sample of cleaned data (first 3 rows):
   Store_ID                   Store_Name  City_ID
0         1       Techstore Alger Centre        1
1         2         Techstore Oran Bahia

#### Cleaning Web Scraped Data

In [7]:
import os
from sklearn.impute import SimpleImputer


print("="*60)
print("CLEANING WEB SCRAPED COMPETITOR DATA")
print("="*60)

# Make sure we have the web scraped data
if 'df_product_prices_competitor' in locals() or 'df_product_prices_competitor' in globals():
    df_web = df_product_prices_competitor.copy()
    print(f"Initial data shape: {df_web.shape}")
    print(f"Columns: {list(df_web.columns)}")
else:
    print("Error: Web scraped data not found. Run web scraping cell first.")
    df_web = pd.DataFrame()

if not df_web.empty:
    # Create directory for cleaned data
    web_cleaned_dir = "Cleaned Web Data"
    os.makedirs(web_cleaned_dir, exist_ok=True)
    
    print("\n1. Checking for duplicates...")
    duplicates_web = df_web.duplicated().sum()
    print(f"   Found {duplicates_web} duplicate(s)")
    df_web_cleaned = df_web.drop_duplicates().copy()
    
    print("\n2. Checking for missing values...")
    missing_counts = df_web_cleaned.isnull().sum()
    for col, count in missing_counts.items():
        if count > 0:
            print(f"   {col}: {count} missing values")
    
    print("\n3. Cleaning product names...")
    # Ensure product_name is string and clean
    df_web_cleaned['product_name'] = df_web_cleaned['product_name'].astype(str).str.strip()
    df_web_cleaned['product_name'] = df_web_cleaned['product_name'].str.title()
    
    print("4. Extracting numeric prices from price strings...")
    
    def extract_numeric_price(price_str):
        """Extract numeric price from string"""
        if pd.isna(price_str):
            return None
        
        price_str = str(price_str)
        
        # Remove currency symbols, commas, spaces
        price_str = price_str.replace('$', '').replace('€', '').replace('£', '')
        price_str = price_str.replace(',', '').replace(' ', '')
        
        # Extract first number found (could be decimal)
        import re
        match = re.search(r'(\d+\.?\d*)', price_str)
        
        if match:
            try:
                return float(match.group(1))
            except:
                return None
        return None
    
    # Apply price extraction
    df_web_cleaned['product_price_numeric'] = df_web_cleaned['product_price'].apply(extract_numeric_price)
    
    print("\n5. Checking extracted prices...")
    missing_prices = df_web_cleaned['product_price_numeric'].isnull().sum()
    print(f"   Successfully extracted prices for {len(df_web_cleaned) - missing_prices}/{len(df_web_cleaned)} products")
    print(f"   Failed to extract prices for {missing_prices} products")
    
    if missing_prices > 0:
        print("\n6. Handling missing prices with SimpleImputer...")
        # Reshape for imputer
        price_data = df_web_cleaned['product_price_numeric'].values.reshape(-1, 1)
        
        # Use SimpleImputer with median (more robust than mean)
        price_imputer = SimpleImputer(strategy='median')
        imputed_prices = price_imputer.fit_transform(price_data)
        
        # Replace with imputed values
        df_web_cleaned['product_price_numeric'] = imputed_prices
        
        print(f"   Filled {missing_prices} missing prices with median: ${price_imputer.statistics_[0]:.2f}")
    else:
        print("\n6. No missing prices to impute.")
    
    print("\n7. Converting prices to integers (no decimals)...")
    df_web_cleaned['product_price_numeric'] = df_web_cleaned['product_price_numeric'].astype(int)
    
    print("\n8. Renaming columns for clarity...")
    df_web_cleaned = df_web_cleaned.rename(columns={
        'product_name': 'Competitor_Product_Name',
        'product_price': 'Competitor_Price_Raw',
        'product_price_numeric': 'Competitor_Price_USD'
    })
    
    # Reorder columns
    df_web_cleaned = df_web_cleaned[['Competitor_Product_Name', 'Competitor_Price_USD', 'Competitor_Price_Raw']]
    
    print("\n9. Saving cleaned data...")
    web_cleaned_path = os.path.join(web_cleaned_dir, "competitor_prices_cleaned.xlsx")
    df_web_cleaned.to_excel(web_cleaned_path, index=False)
    
    print("\n" + "-"*50)
    print("WEB DATA CLEANING SUMMARY")
    print("-"*50)
    print(f"Original records: {df_web.shape[0]}")
    print(f"After removing duplicates: {df_web_cleaned.shape[0]}")
    print(f"Price statistics (USD):")
    print(f"  Min: ${df_web_cleaned['Competitor_Price_USD'].min():,}")
    print(f"  Max: ${df_web_cleaned['Competitor_Price_USD'].max():,}")
    print(f"  Avg: ${df_web_cleaned['Competitor_Price_USD'].mean():,.0f}")
    print(f"  Total products: {len(df_web_cleaned)}")
    print(f"\nSaved to: {web_cleaned_path}")
    
    # Display sample
    print("\nSample of cleaned competitor prices (first 5 rows):")
    print(df_web_cleaned.head())


CLEANING WEB SCRAPED COMPETITOR DATA
Initial data shape: (38, 2)
Columns: ['product_name', 'product_price']

1. Checking for duplicates...
   Found 0 duplicate(s)

2. Checking for missing values...

3. Cleaning product names...
4. Extracting numeric prices from price strings...

5. Checking extracted prices...
   Successfully extracted prices for 38/38 products
   Failed to extract prices for 0 products

6. No missing prices to impute.

7. Converting prices to integers (no decimals)...

8. Renaming columns for clarity...

9. Saving cleaned data...

--------------------------------------------------
WEB DATA CLEANING SUMMARY
--------------------------------------------------
Original records: 38
After removing duplicates: 38
Price statistics (USD):
  Min: $930
  Max: $350,400
  Avg: $89,032
  Total products: 38

Saved to: Cleaned Web Data\competitor_prices_cleaned.xlsx

Sample of cleaned competitor prices (first 5 rows):
      Competitor_Product_Name  Competitor_Price_USD Competitor_Pri

#### Cleaning OCR Data

In [ ]:
# ⚠️⚠️⚠️⚠️ i didnt execute this cell because tesseract doesnt work in my device, so its cleaned excells file dont appear
print("="*60)
print("CLEANING OCR DATA FROM LEGACY INVOICES")
print("="*60)

# Make sure we have the OCR data
if 'df_invoices' in locals() or 'df_invoices' in globals():
    df_ocr = df_invoices.copy()
    print(f"Initial OCR data shape: {df_ocr.shape}")
    print(f"Columns: {list(df_ocr.columns)}")
else:
    print("Error: OCR data not found. Run OCR extraction cell first.")
    df_ocr = pd.DataFrame()

if not df_ocr.empty:
    # Create directory for cleaned OCR data
    ocr_cleaned_dir = "Cleaned OCR Data"
    os.makedirs(ocr_cleaned_dir, exist_ok=True)
    
    print("\nInitial data quality check:")
    print(f"Total records: {len(df_ocr)}")
    print("\nMissing values by column:")
    missing_counts = df_ocr.isnull().sum()
    for col, count in missing_counts.items():
        print(f"  {col}: {count} missing")
    
    print("\n1. Removing duplicates...")
    duplicates_ocr = df_ocr.duplicated().sum()
    print(f"   Found {duplicates_ocr} duplicate(s)")
    df_ocr_cleaned = df_ocr.drop_duplicates().copy()
    
    print("\n2. Standardizing Date column...")
    def clean_ocr_date(date_value):
        """Clean and standardize OCR extracted dates"""
        if pd.isna(date_value) or str(date_value).strip().lower() in ['not found', 'nan', 'none', '']:
            return None
        
        date_str = str(date_value).strip()
        
        # Try to parse common date formats
        import re
        
        # Pattern 1: YYYY-MM-DD
        match1 = re.match(r'(\d{4})-(\d{1,2})-(\d{1,2})', date_str)
        if match1:
            year, month, day = match1.groups()
            return f"{year}-{int(month):02d}-{int(day):02d}"
        
        # Pattern 2: DD/MM/YYYY
        match2 = re.match(r'(\d{1,2})/(\d{1,2})/(\d{4})', date_str)
        if match2:
            day, month, year = match2.groups()
            return f"{year}-{int(month):02d}-{int(day):02d}"
        
        # Pattern 3: DD-MM-YYYY
        match3 = re.match(r'(\d{1,2})-(\d{1,2})-(\d{4})', date_str)
        if match3:
            day, month, year = match3.groups()
            return f"{year}-{int(month):02d}-{int(day):02d}"
        
        # If no pattern matches, return None
        return None
    
    df_ocr_cleaned['Date'] = df_ocr_cleaned['Date'].apply(clean_ocr_date)
    
    print("\n3. Cleaning CustomerID...")
    df_ocr_cleaned['CustomerID'] = df_ocr_cleaned['CustomerID'].astype(str).str.strip().str.upper()
    # Replace "Not found" with UNKNOWN
    df_ocr_cleaned.loc[df_ocr_cleaned['CustomerID'].str.contains('not found', case=False, na=False), 'CustomerID'] = 'UNKNOWN'
    
    print("\n4. Cleaning Product Name...")
    df_ocr_cleaned['Product Name'] = df_ocr_cleaned['Product Name'].astype(str).str.strip()
    df_ocr_cleaned['Product Name'] = df_ocr_cleaned['Product Name'].str.title()
    # Replace "Not found" with UNKNOWN_PRODUCT
    df_ocr_cleaned.loc[df_ocr_cleaned['Product Name'].str.contains('not found', case=False, na=False), 'Product Name'] = 'UNKNOWN_PRODUCT'
    
    print("\n5. Cleaning numeric columns (Quantity, Total Revenue)...")
    
    def clean_numeric_value(value):
        """Extract integer from potentially messy OCR output"""
        if pd.isna(value) or str(value).strip().lower() in ['not found', 'nan', 'none', '']:
            return None
        
        value_str = str(value).strip()
        
        # Extract first number found
        import re
        match = re.search(r'(\d+)', value_str)
        
        if match:
            try:
                return int(match.group(1))
            except:
                return None
        return None
    
    # Apply to Quantity and Total Revenue
    df_ocr_cleaned['Quantity'] = df_ocr_cleaned['Quantity'].apply(clean_numeric_value)
    df_ocr_cleaned['Total Revenue'] = df_ocr_cleaned['Total Revenue'].apply(clean_numeric_value)
    
    print("\n6. Handling missing values with SimpleImputer...")
    
    # Check which numeric columns have missing values
    numeric_cols = ['Quantity', 'Total Revenue']
    for col in numeric_cols:
        missing_count = df_ocr_cleaned[col].isnull().sum()
        if missing_count > 0:
            print(f"   {col}: {missing_count} missing values")
            
            # Use SimpleImputer with median
            data = df_ocr_cleaned[col].values.reshape(-1, 1)
            imputer = SimpleImputer(strategy='median')
            imputed_data = imputer.fit_transform(data)
            df_ocr_cleaned[col] = imputed_data
            
            print(f"     Filled with median: {imputer.statistics_[0]:.0f}")
        else:
            print(f"   {col}: No missing values")
    
    # Convert to integers
    df_ocr_cleaned['Quantity'] = df_ocr_cleaned['Quantity'].astype(int)
    df_ocr_cleaned['Total Revenue'] = df_ocr_cleaned['Total Revenue'].astype(int)
    
    print("\n7. Handling missing dates...")
    missing_dates = df_ocr_cleaned['Date'].isnull().sum()
    if missing_dates > 0:
        print(f"   Found {missing_dates} missing dates")
        
        # Find most common date (mode)
        date_mode = df_ocr_cleaned['Date'].mode()
        if not date_mode.empty:
            fill_date = date_mode.iloc[0]
            df_ocr_cleaned['Date'] = df_ocr_cleaned['Date'].fillna(fill_date)
            print(f"   Filled with most common date: {fill_date}")
        else:
            # If no mode, use a default
            df_ocr_cleaned['Date'] = df_ocr_cleaned['Date'].fillna('2022-01-01')
            print(f"   Filled with default date: 2022-01-01")
    else:
        print("   No missing dates found")
    
    print("\n8. Final data validation...")
    print(f"   Final shape: {df_ocr_cleaned.shape}")
    print(f"   Missing values after cleaning:")
    for col in df_ocr_cleaned.columns:
        missing = df_ocr_cleaned[col].isnull().sum()
        if missing > 0:
            print(f"     {col}: {missing} missing")
    
    print("\n9. Saving cleaned OCR data...")
    ocr_cleaned_path = os.path.join(ocr_cleaned_dir, "legacy_invoices_cleaned.xlsx")
    df_ocr_cleaned.to_excel(ocr_cleaned_path, index=False)
    
    print("\n" + "-"*50)
    print("OCR DATA CLEANING SUMMARY")
    print("-"*50)
    print(f"Original records: {df_ocr.shape[0]}")
    print(f"After removing duplicates: {df_ocr_cleaned.shape[0]}")
    print(f"Date range: {df_ocr_cleaned['Date'].min()} to {df_ocr_cleaned['Date'].max()}")
    print(f"Unique customers: {df_ocr_cleaned['CustomerID'].nunique()}")
    print(f"Unique products: {df_ocr_cleaned['Product Name'].nunique()}")
    print(f"Total quantity sold: {df_ocr_cleaned['Quantity'].sum()}")
    print(f"Total revenue: {df_ocr_cleaned['Total Revenue'].sum():,}")
    print(f"\nSaved to: {ocr_cleaned_path}")
    
    # Display sample
    print("\nSample of cleaned OCR data (first 5 rows):")
    print(df_ocr_cleaned.head())


#### Excell files CLEANING, TRANSFORMATION & CURRENCY CONVERSION

In [10]:
import os
import pandas as pd
import numpy as np
import re
from datetime import datetime
from sklearn.impute import SimpleImputer

# Create directory for cleaned files
cleaned_dir = "Cleaned Excel Files"
os.makedirs(cleaned_dir, exist_ok=True)

# Function to clean text data
def clean_text_column(series):
    """Clean text columns: trim whitespace, fix case sensitivity"""
    if series.dtype == 'object':
        # Trim whitespace
        series = series.str.strip()
        # Capitalize first letter of each word for consistency
        series = series.str.title()
    return series

# Function to parse dates in various formats
def parse_date_flexible(date_str):
    """Parse dates from various formats including month-year"""
    if pd.isna(date_str):
        return None
    
    date_str = str(date_str).strip()
    
    # Handle month-year formats like "Feb-2023", "Jan-24"
    month_year_pattern = r'^([A-Za-z]{3,})-(\d{2,4})$'
    month_year_match = re.match(month_year_pattern, date_str)
    
    if month_year_match:
        month_str = month_year_match.group(1)
        year_str = month_year_match.group(2)
        
        # Convert 2-digit year to 4-digit
        if len(year_str) == 2:
            year_str = '20' + year_str
        
        # Map month abbreviations to numbers
        month_map = {
            'jan': 1, 'feb': 2, 'mar': 3, 'apr': 4, 'may': 5, 'jun': 6,
            'jul': 7, 'aug': 8, 'sep': 9, 'oct': 10, 'nov': 11, 'dec': 12
        }
        
        month_num = month_map.get(month_str.lower(), 1)  # Default to January if not found
        return f"{year_str}-{month_num:02d}-01"
    
    # Handle DD/MM/YYYY format like "01/03/2025"
    if re.match(r'^\d{1,2}/\d{1,2}/\d{4}$', date_str):
        day, month, year = date_str.split('/')
        return f"{year}-{int(month):02d}-{int(day):02d}"
    
    # Handle YYYY-MM-DD format (with or without time)
    if re.match(r'^\d{4}-\d{1,2}-\d{1,2}', date_str):
        try:
            dt = pd.to_datetime(date_str, errors='coerce')
            return dt.strftime('%Y-%m-%d') if not pd.isna(dt) else None
        except:
            return None
    
    return None

# Function to standardize date format
def standardize_dates(df, date_columns):
    """Convert various date formats to YYYY-MM-DD"""
    for col in date_columns:
        if col in df.columns:
            # Apply flexible date parsing
            df[col] = df[col].apply(parse_date_flexible)
            
            # For any remaining dates, try pandas to_datetime as fallback
            df[col] = pd.to_datetime(df[col], errors='coerce')
            df[col] = df[col].dt.strftime('%Y-%m-%d')
    return df

# ==============================================
# 1. Clean marketing_expenses.xlsx
# ==============================================
print("="*60)
print("CLEANING MARKETING EXPENSES")
print("="*60)

# Check for duplicates
duplicates_marketing = df_marketing_expenses.duplicated().sum()
print(f"Found {duplicates_marketing} duplicate(s) in marketing data")

# Remove duplicates
df_marketing_cleaned = df_marketing_expenses.drop_duplicates().copy()

# Check date formats before cleaning
print("\nSample of original date formats:")
print(df_marketing_cleaned['Date'].head(10).to_string())

# Standardize date format with improved parsing
df_marketing_cleaned = standardize_dates(df_marketing_cleaned, ['Date'])

# Clean text columns
text_cols_marketing = ['Category', 'Campaign_Type']
for col in text_cols_marketing:
    if col in df_marketing_cleaned.columns:
        df_marketing_cleaned[col] = clean_text_column(df_marketing_cleaned[col])

# Step 1: Detect and report invalid values in Marketing_Cost_USD
print("\n" + "-"*50)
print("DETECTING INVALID VALUES IN Marketing_Cost_USD")
print("-"*50)

# Ensure Marketing_Cost_USD is numeric
df_marketing_cleaned['Marketing_Cost_USD'] = pd.to_numeric(df_marketing_cleaned['Marketing_Cost_USD'], errors='coerce')

# Count missing values
missing_values = df_marketing_cleaned['Marketing_Cost_USD'].isnull().sum()
print(f"Missing values: {missing_values}")

# Count negative values (invalid costs)
negative_values = (df_marketing_cleaned['Marketing_Cost_USD'] < 0).sum()
print(f"Negative values (invalid costs): {negative_values}")

# Count zero values
zero_values = (df_marketing_cleaned['Marketing_Cost_USD'] == 0).sum()
print(f"Zero values: {zero_values}")

# Display rows with negative values
if negative_values > 0:
    print("\nRows with negative Marketing_Cost_USD (invalid):")
    negative_rows = df_marketing_cleaned[df_marketing_cleaned['Marketing_Cost_USD'] < 0]
    print(negative_rows[['Date', 'Category', 'Marketing_Cost_USD']].head())
# Step 2: Handle negative values - convert to positive (remove minus sign)
print("\n" + "-"*50)
print("HANDLING NEGATIVE VALUES")
print("-"*50)

# Store original values for negative rows
original_negatives = df_marketing_cleaned[df_marketing_cleaned['Marketing_Cost_USD'] < 0]['Marketing_Cost_USD'].copy()

# Convert negative values to positive (remove minus sign)
df_marketing_cleaned.loc[df_marketing_cleaned['Marketing_Cost_USD'] < 0, 'Marketing_Cost_USD'] = \
    df_marketing_cleaned.loc[df_marketing_cleaned['Marketing_Cost_USD'] < 0, 'Marketing_Cost_USD'].abs()
print(f"Converted {negative_values} negative values to positive")

# Step 3: Handle missing values using SimpleImputer
print("\n" + "-"*50)
print("HANDLING MISSING VALUES WITH SIMPLEIMPUTER")
print("-"*50)

# Create a copy of the column for imputation
cost_data = df_marketing_cleaned['Marketing_Cost_USD'].values.reshape(-1, 1)

# Initialize SimpleImputer with mean strategy
imputer = SimpleImputer(strategy='mean')

# Fit and transform the data
imputed_data = imputer.fit_transform(cost_data)

# Replace the original column with imputed values
df_marketing_cleaned['Marketing_Cost_USD'] = imputed_data

print(f"Filled {missing_values} missing values with mean: {imputer.statistics_[0]:.2f}")

# Step 4: Apply currency conversion (USD to DZD)
print("\n" + "-"*50)
print("APPLYING CURRENCY CONVERSION (USD to DZD)")
print("-"*50)

# Current exchange rate (1 USD = 134.5 DZD as approximate rate)
exchange_rate = 134.5  # 1 USD = 134.5 DZD

# Create new column for marketing cost in DZD
df_marketing_cleaned['Marketing_Cost_DZD'] = df_marketing_cleaned['Marketing_Cost_USD'] * exchange_rate

# Convert to integers (no decimal places)
df_marketing_cleaned['Marketing_Cost_USD'] = df_marketing_cleaned['Marketing_Cost_USD'].astype(int)
df_marketing_cleaned['Marketing_Cost_DZD'] = df_marketing_cleaned['Marketing_Cost_DZD'].astype(int)

print(f"Using exchange rate: 1 USD = {exchange_rate} DZD")
print(f"Total marketing cost in USD: ${df_marketing_cleaned['Marketing_Cost_USD'].sum():,.0f}")
print(f"Total marketing cost in DZD: {df_marketing_cleaned['Marketing_Cost_DZD'].sum():,.0f} DZD")

# Save the fully cleaned and transformed marketing data
marketing_cleaned_path = os.path.join(cleaned_dir, "marketing_expenses_cleaned.xlsx")
df_marketing_cleaned.to_excel(marketing_cleaned_path, index=False)
print(f"\nSaved cleaned marketing data to: {marketing_cleaned_path}")

# ==============================================
# 2. Clean monthly_targets.xlsx
# ==============================================
print("\n" + "="*60)
print("CLEANING MONTHLY TARGETS")
print("="*60)

# Check for duplicates
duplicates_targets = df_monthly_targets.duplicated().sum()
print(f"Found {duplicates_targets} duplicate(s) in targets data")

# Remove duplicates
df_targets_cleaned = df_monthly_targets.drop_duplicates().copy()

# Clean Store_ID column - extract numeric part from strings like "S1", "Store_5", etc.
def clean_store_id(store_id):
    if pd.isna(store_id):
        return None
    # Extract digits from string
    store_str = str(store_id)
    digits = re.findall(r'\d+', store_str)
    return int(digits[0]) if digits else None

df_targets_cleaned['Store_ID'] = df_targets_cleaned['Store_ID'].apply(clean_store_id)

# Standardize date format
df_targets_cleaned = standardize_dates(df_targets_cleaned, ['Month'])

# Clean Manager_Name text
df_targets_cleaned['Manager_Name'] = clean_text_column(df_targets_cleaned['Manager_Name'])

# Clean Target_Revenue - remove commas and convert to numeric
if 'Target_Revenue' in df_targets_cleaned.columns:
    # Convert to string first to handle mixed types
    df_targets_cleaned['Target_Revenue'] = df_targets_cleaned['Target_Revenue'].astype(str)
    # Remove commas and non-numeric characters
    df_targets_cleaned['Target_Revenue'] = df_targets_cleaned['Target_Revenue'].str.replace(',', '').str.replace(' ', '')
    # Convert to numeric, coerce errors to NaN
    df_targets_cleaned['Target_Revenue'] = pd.to_numeric(df_targets_cleaned['Target_Revenue'], errors='coerce')
    
    # Check for missing or invalid values
    missing_revenue = df_targets_cleaned['Target_Revenue'].isnull().sum()
    negative_revenue = (df_targets_cleaned['Target_Revenue'] < 0).sum()
    print(f"Found {missing_revenue} missing target revenue values")
    print(f"Found {negative_revenue} negative target revenue values (invalid)")
    
    # Handle missing values using SimpleImputer
    if missing_revenue > 0:
        print("\nHandling missing Target_Revenue values with SimpleImputer...")
        revenue_data = df_targets_cleaned['Target_Revenue'].values.reshape(-1, 1)
        revenue_imputer = SimpleImputer(strategy='mean')
        imputed_revenue = revenue_imputer.fit_transform(revenue_data)
        df_targets_cleaned['Target_Revenue'] = imputed_revenue
        print(f"Filled {missing_revenue} missing values with mean: {revenue_imputer.statistics_[0]:,.2f}")
    
    # Handle negative values - convert to positive (remove minus sign)
    if negative_revenue > 0:
        print(f"\nConverting {negative_revenue} negative target revenue values to positive...")
        df_targets_cleaned.loc[df_targets_cleaned['Target_Revenue'] < 0, 'Target_Revenue'] = \
            df_targets_cleaned.loc[df_targets_cleaned['Target_Revenue'] < 0, 'Target_Revenue'].abs()
    
    # Convert to integers
    df_targets_cleaned['Target_Revenue'] = df_targets_cleaned['Target_Revenue'].astype(int)

# Check for missing values in Store_ID and Manager_Name
missing_store_id = df_targets_cleaned['Store_ID'].isnull().sum()
missing_manager_name = df_targets_cleaned['Manager_Name'].isnull().sum()
missing_month = df_targets_cleaned['Month'].isnull().sum()

print(f"\nOther missing values:")
print(f"  - Missing Store_ID: {missing_store_id}")
print(f"  - Missing Manager_Name: {missing_manager_name}")
print(f"  - Missing Month: {missing_month}")

# Handle missing Store_ID (if any)
if missing_store_id > 0:
    print("\nHandling missing Store_ID values...")
    # For missing Store_ID, we can fill with the most frequent value
    most_frequent_store = df_targets_cleaned['Store_ID'].mode()[0]
    df_targets_cleaned['Store_ID'] = df_targets_cleaned['Store_ID'].fillna(most_frequent_store)
    print(f"Filled {missing_store_id} missing Store_ID values with: {most_frequent_store}")

# Handle missing Manager_Name (if any)
if missing_manager_name > 0:
    print("\nHandling missing Manager_Name values...")
    # Fill missing manager names with "Unknown"
    df_targets_cleaned['Manager_Name'] = df_targets_cleaned['Manager_Name'].fillna("Unknown")

# Handle missing Month (if any)
if missing_month > 0:
    print("\nHandling missing Month values...")
    # Fill missing months with the most frequent month
    most_frequent_month = df_targets_cleaned['Month'].mode()[0]
    df_targets_cleaned['Month'] = df_targets_cleaned['Month'].fillna(most_frequent_month)
    print(f"Filled {missing_month} missing Month values with: {most_frequent_month}")

# Save cleaned file
targets_cleaned_path = os.path.join(cleaned_dir, "monthly_targets_cleaned.xlsx")
df_targets_cleaned.to_excel(targets_cleaned_path, index=False)
print(f"\nSaved cleaned targets data to: {targets_cleaned_path}")

# ==============================================
# 3. Clean shipping_rates.xlsx
# ==============================================
print("\n" + "="*60)
print("CLEANING SHIPPING RATES")
print("="*60)

# Check for duplicates
duplicates_shipping = df_shipping_rates.duplicated().sum()
print(f"Found {duplicates_shipping} duplicate(s) in shipping data")

# Remove duplicates
df_shipping_cleaned = df_shipping_rates.drop_duplicates().copy()

# Clean text columns
text_cols_shipping = ['region_name', 'provider']
for col in text_cols_shipping:
    if col in df_shipping_cleaned.columns:
        df_shipping_cleaned[col] = clean_text_column(df_shipping_cleaned[col])

# Check for missing values
missing_shipping_cost = df_shipping_cleaned['shipping_cost'].isnull().sum()
missing_delivery_days = df_shipping_cleaned['average_delivery_days'].isnull().sum()
print(f"Found {missing_shipping_cost} missing shipping costs")
print(f"Found {missing_delivery_days} missing average delivery days")

# Check for negative values (invalid for shipping)
negative_shipping = (df_shipping_cleaned['shipping_cost'] < 0).sum()
negative_days = (df_shipping_cleaned['average_delivery_days'] < 0).sum()
print(f"Found {negative_shipping} negative shipping costs (invalid)")
print(f"Found {negative_days} negative delivery days (invalid)")

# Handle missing and negative values
if missing_shipping_cost > 0 or negative_shipping > 0 or missing_delivery_days > 0 or negative_days > 0:
    print("\nHandling missing and invalid values in shipping data...")
    
    # Handle shipping_cost
    if missing_shipping_cost > 0 or negative_shipping > 0:
        shipping_data = df_shipping_cleaned['shipping_cost'].values.reshape(-1, 1)
        shipping_imputer = SimpleImputer(strategy='mean')
        imputed_shipping = shipping_imputer.fit_transform(shipping_data)
        df_shipping_cleaned['shipping_cost'] = imputed_shipping
        
        # Replace any remaining negative values with mean
        df_shipping_cleaned.loc[df_shipping_cleaned['shipping_cost'] < 0, 'shipping_cost'] = shipping_imputer.statistics_[0]
        
        print(f"Filled {missing_shipping_cost} missing shipping costs and fixed {negative_shipping} negative values")
    
    # Handle average_delivery_days
    if missing_delivery_days > 0 or negative_days > 0:
        days_data = df_shipping_cleaned['average_delivery_days'].values.reshape(-1, 1)
        days_imputer = SimpleImputer(strategy='mean')
        imputed_days = days_imputer.fit_transform(days_data)
        df_shipping_cleaned['average_delivery_days'] = imputed_days
        
        # Replace any remaining negative values with mean
        df_shipping_cleaned.loc[df_shipping_cleaned['average_delivery_days'] < 0, 'average_delivery_days'] = days_imputer.statistics_[0]
        
        print(f"Filled {missing_delivery_days} missing delivery days and fixed {negative_days} negative values")
    
    # Convert to integers
    df_shipping_cleaned['shipping_cost'] = df_shipping_cleaned['shipping_cost'].astype(int)
    df_shipping_cleaned['average_delivery_days'] = df_shipping_cleaned['average_delivery_days'].astype(int)

# Save cleaned file
shipping_cleaned_path = os.path.join(cleaned_dir, "shipping_rates_cleaned.xlsx")
df_shipping_cleaned.to_excel(shipping_cleaned_path, index=False)
print(f"Saved cleaned shipping data to: {shipping_cleaned_path}")


CLEANING MARKETING EXPENSES
Found 0 duplicate(s) in marketing data

Sample of original date formats:
0    2023-01-01 00:00:00
1    2023-01-01 00:00:00
2    2023-01-01 00:00:00
3    2023-01-01 00:00:00
4    2023-01-01 00:00:00
5    2023-02-01 00:00:00
6    2023-02-01 00:00:00
7    2023-02-01 00:00:00
8    2023-02-01 00:00:00
9    2023-02-01 00:00:00

--------------------------------------------------
DETECTING INVALID VALUES IN Marketing_Cost_USD
--------------------------------------------------
Missing values: 4
Negative values (invalid costs): 5
Zero values: 0

Rows with negative Marketing_Cost_USD (invalid):
           Date     Category  Marketing_Cost_USD
31   2023-07-01  Smartphones             -2905.0
32   2023-07-01        Audio              -453.0
96   2024-08-01  Smartphones             -1700.0
140  2025-05-01    Computers             -1323.0
159  2025-08-01     Printers             -1968.0

--------------------------------------------------
HANDLING NEGATIVE VALUES
----------

####  Sentiment Analysis using VADER

In [1]:
# ⚠️⚠️⚠️ Installing vaderSentiment required, use: pip install vaderSentiment

from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
import pandas as pd
import os

# Initialize VADER sentiment analyzer
analyzer = SentimentIntensityAnalyzer()

# Load cleaned reviews data
print("\n1. Loading customer reviews data...")
reviews_file = "Cleaned ERP Data/table_reviews_cleaned.xlsx"

try:
    df_reviews = pd.read_excel(reviews_file)
    print(f"   ✓ Loaded {len(df_reviews)} reviews from {reviews_file}")
    print(f"   Columns: {list(df_reviews.columns)}")
except FileNotFoundError:
    print(f"   ✗ Error: File not found - {reviews_file}")
    print("   Make sure you ran the ERP data cleaning cell first!")
    df_reviews = pd.DataFrame()

if not df_reviews.empty:
    # Display sample of original data
    print("\n2. Sample of original reviews:")
    print(df_reviews[['Product_ID', 'Rating', 'Review_Text']].head(3))
    
    # Function to get sentiment score
    def get_sentiment_score(text):
        """
        Analyze sentiment of text using VADER
        Returns compound score between -1.0 (most negative) and +1.0 (most positive)
        """
        if pd.isna(text) or str(text).strip() == '':
            return 0.0  # Neutral for empty reviews
        
        try:
            # Get sentiment scores
            scores = analyzer.polarity_scores(str(text))
            # Return compound score (ranges from -1 to +1)
            return scores['compound']
        except:
            return 0.0  # Return neutral if analysis fails
    
    print("\n3. Analyzing sentiment for each review...")
    # Apply sentiment analysis to each review
    df_reviews['Sentiment_Score'] = df_reviews['Review_Text'].apply(get_sentiment_score)
    
    print(f"   ✓ Sentiment analysis complete for {len(df_reviews)} reviews")
    
    # Display statistics
    print("\n4. Sentiment Analysis Statistics:")
    print(f"   Mean Sentiment Score: {df_reviews['Sentiment_Score'].mean():.3f}")
    print(f"   Min Sentiment Score: {df_reviews['Sentiment_Score'].min():.3f}")
    print(f"   Max Sentiment Score: {df_reviews['Sentiment_Score'].max():.3f}")
    print(f"   Median Sentiment Score: {df_reviews['Sentiment_Score'].median():.3f}")
    
    # Classify sentiments
    def classify_sentiment(score):
        """Classify sentiment into categories"""
        if score >= 0.05:
            return 'Positive'
        elif score <= -0.05:
            return 'Negative'
        else:
            return 'Neutral'
    
    df_reviews['Sentiment_Category'] = df_reviews['Sentiment_Score'].apply(classify_sentiment)
    
    # Count sentiments by category
    sentiment_counts = df_reviews['Sentiment_Category'].value_counts()
    print("\n5. Sentiment Distribution:")
    for category, count in sentiment_counts.items():
        percentage = (count / len(df_reviews)) * 100
        print(f"   {category}: {count} ({percentage:.1f}%)")
    
    # Calculate average sentiment per product
    print("\n6. Calculating average sentiment per product...")
    product_sentiment = df_reviews.groupby('Product_ID').agg({
        'Sentiment_Score': ['mean', 'count'],
        'Rating': 'mean'
    }).reset_index()
    
    # Flatten column names
    product_sentiment.columns = ['Product_ID', 'Avg_Sentiment_Score', 'Review_Count', 'Avg_Rating']
    
    # Round to 3 decimal places
    product_sentiment['Avg_Sentiment_Score'] = product_sentiment['Avg_Sentiment_Score'].round(3)
    product_sentiment['Avg_Rating'] = product_sentiment['Avg_Rating'].round(2)
    
    print(f"   ✓ Calculated sentiment for {len(product_sentiment)} unique products")
    
    # Display top 10 products by sentiment
    print("\n7. Top 10 Products by Sentiment Score:")
    top_products = product_sentiment.nlargest(10, 'Avg_Sentiment_Score')
    print(top_products.to_string(index=False))
    
    # Display bottom 10 products by sentiment
    print("\n8. Bottom 10 Products by Sentiment Score:")
    bottom_products = product_sentiment.nsmallest(10, 'Avg_Sentiment_Score')
    print(bottom_products.to_string(index=False))
    
    # Save results
    print("\n9. Saving sentiment analysis results...")
    
    # Create output directory
    sentiment_dir = "Sentiment Analysis Results"
    os.makedirs(sentiment_dir, exist_ok=True)
    
    # Save detailed reviews with sentiment scores
    reviews_sentiment_path = os.path.join(sentiment_dir, "reviews_with_sentiment.xlsx")
    df_reviews.to_excel(reviews_sentiment_path, index=False)
    print(f"   ✓ Saved detailed reviews to: {reviews_sentiment_path}")
    
    # Save product-level sentiment aggregation
    product_sentiment_path = os.path.join(sentiment_dir, "product_sentiment_scores.xlsx")
    product_sentiment.to_excel(product_sentiment_path, index=False)
    print(f"   ✓ Saved product sentiment scores to: {product_sentiment_path}")
    
    # Display sample of results
    print("\n10. Sample of reviews with sentiment scores:")
    sample_cols = ['Product_ID', 'Rating', 'Review_Text', 'Sentiment_Score', 'Sentiment_Category']
    print(df_reviews[sample_cols].head(5))
    
    print("\n" + "="*60)
    print("SENTIMENT ANALYSIS COMPLETE")
    print("="*60)
    print(f"✓ Analyzed {len(df_reviews)} individual reviews")
    print(f"✓ Generated sentiment scores for {len(product_sentiment)} products")
    print(f"✓ Results saved to: {sentiment_dir}/")

else:
    print("\n✗ No review data available. Cannot perform sentiment analysis.")

SENTIMENT ANALYSIS USING VADER

1. Loading customer reviews data...
   ✓ Loaded 3000 reviews from Cleaned ERP Data/table_reviews_cleaned.xlsx
   Columns: ['Review_ID', 'Product_ID', 'Customer_ID', 'Rating', 'Review_Text']

2. Sample of original reviews:
  Product_ID  Rating                                   Review_Text
0       P127       3        Average, expected better battery life.
1       P103       5        Excellent product, highly recommended!
2       P102       4  Good, but shipping was a bit slow to Guelma.

3. Analyzing sentiment for each review...
   ✓ Sentiment analysis complete for 3000 reviews

4. Sentiment Analysis Statistics:
   Mean Sentiment Score: 0.170
   Min Sentiment Score: -0.801
   Max Sentiment Score: 0.765
   Median Sentiment Score: 0.238

5. Sentiment Distribution:
   Positive: 1932 (64.4%)
   Negative: 1068 (35.6%)

6. Calculating average sentiment per product...
   ✓ Calculated sentiment for 38 unique products

7. Top 10 Products by Sentiment Score:
Product

#### Net Profit Calculation

In [2]:
import pandas as pd
import numpy as np
import os

# Load all necessary cleaned data files
print("\n1. Loading cleaned data files...")

# Load sales data
try:
    df_sales = pd.read_excel("Cleaned ERP Data/table_sales_cleaned.xlsx")
    print(f"   ✓ Loaded sales data: {len(df_sales)} transactions")
except:
    print("   ✗ Error loading sales data")
    df_sales = pd.DataFrame()

# Load products data
try:
    df_products = pd.read_excel("Cleaned ERP Data/table_products_cleaned.xlsx")
    print(f"   ✓ Loaded products data: {len(df_products)} products")
except:
    print("   ✗ Error loading products data")
    df_products = pd.DataFrame()

# Load stores data
try:
    df_stores = pd.read_excel("Cleaned ERP Data/table_stores_cleaned.xlsx")
    print(f"   ✓ Loaded stores data: {len(df_stores)} stores")
except:
    print("   ✗ Error loading stores data")
    df_stores = pd.DataFrame()

# Load cities data
try:
    df_cities = pd.read_excel("Cleaned ERP Data/table_cities_cleaned.xlsx")
    print(f"   ✓ Loaded cities data: {len(df_cities)} cities")
except:
    print("   ✗ Error loading cities data")
    df_cities = pd.DataFrame()

# Load categories and subcategories
try:
    df_categories = pd.read_excel("Cleaned ERP Data/table_categories_cleaned.xlsx")
    df_subcategories = pd.read_excel("Cleaned ERP Data/table_subcategories_cleaned.xlsx")
    print(f"   ✓ Loaded category data: {len(df_categories)} categories, {len(df_subcategories)} subcategories")
except:
    print("   ✗ Error loading category data")
    df_categories = pd.DataFrame()
    df_subcategories = pd.DataFrame()

# Load external files (Marketing, Shipping)
try:
    df_marketing = pd.read_excel("Cleaned Excel Files/marketing_expenses_cleaned.xlsx")
    print(f"   ✓ Loaded marketing data: {len(df_marketing)} records")
except:
    print("   ✗ Error loading marketing data")
    df_marketing = pd.DataFrame()

try:
    df_shipping = pd.read_excel("Cleaned Excel Files/shipping_rates_cleaned.xlsx")
    print(f"   ✓ Loaded shipping rates: {len(df_shipping)} records")
except:
    print("   ✗ Error loading shipping rates")
    df_shipping = pd.DataFrame()

# Check if all required data is loaded
if df_sales.empty or df_products.empty:
    print("\n✗ Error: Missing critical data (sales or products). Cannot calculate net profit.")
else:
    print("\n2. Preparing data for net profit calculation...")
    
    # Start with sales data
    df_profit = df_sales.copy()
    
    print(f"   Initial sales records: {len(df_profit)}")
    
    # Step 1: Merge with products to get Unit_Cost
    print("\n3. Merging with product costs...")
    df_profit = df_profit.merge(
        df_products[['Product_ID', 'Unit_Cost', 'SubCat_ID']],
        on='Product_ID',
        how='left'
    )
    
    # Check for missing product costs
    missing_costs = df_profit['Unit_Cost'].isnull().sum()
    if missing_costs > 0:
        print(f"   ⚠ Warning: {missing_costs} sales missing product cost. Filling with 0.")
        df_profit['Unit_Cost'] = df_profit['Unit_Cost'].fillna(0)
    
    print(f"   ✓ Product costs merged")
    
    # Step 2: Get category information for marketing costs
    print("\n4. Linking products to categories...")
    if not df_subcategories.empty and not df_categories.empty:
        df_profit = df_profit.merge(
            df_subcategories[['SubCat_ID', 'Category_ID']],
            on='SubCat_ID',
            how='left'
        )
        df_profit = df_profit.merge(
            df_categories[['Category_ID', 'Category_Name']],
            on='Category_ID',
            how='left'
        )
        print(f"   ✓ Category information added")
    else:
        print("   ⚠ Category data not available, marketing costs cannot be allocated")
        df_profit['Category_Name'] = 'Unknown'
    
    # Step 3: Get region for shipping costs
    print("\n5. Linking stores to regions for shipping costs...")
    if not df_stores.empty and not df_cities.empty:
        df_profit = df_profit.merge(
            df_stores[['Store_ID', 'City_ID']],
            on='Store_ID',
            how='left'
        )
        df_profit = df_profit.merge(
            df_cities[['City_ID', 'Region']],
            on='City_ID',
            how='left',
            suffixes=('', '_city')
        )
        print(f"   ✓ Region information added")
    else:
        print("   ⚠ Store/city data not available, shipping costs cannot be allocated")
        df_profit['Region'] = 'Unknown'
    
    # Step 4: Add shipping costs based on region
    print("\n6. Calculating shipping costs per transaction...")
    if not df_shipping.empty:
        # Calculate average shipping cost per region
        shipping_avg = df_shipping.groupby('region_name')['shipping_cost'].mean().reset_index()
        shipping_avg.columns = ['Region', 'Shipping_Cost']
        
        df_profit = df_profit.merge(
            shipping_avg,
            on='Region',
            how='left'
        )
        
        # Fill missing shipping costs with overall average
        overall_avg_shipping = df_shipping['shipping_cost'].mean()
        df_profit['Shipping_Cost'] = df_profit['Shipping_Cost'].fillna(overall_avg_shipping)
        
        print(f"   ✓ Shipping costs added (avg: {df_profit['Shipping_Cost'].mean():.2f} DZD)")
    else:
        print("   ⚠ Shipping data not available, setting shipping cost to 0")
        df_profit['Shipping_Cost'] = 0
    
    # Step 5: Add marketing costs
    print("\n7. Calculating marketing costs per transaction...")
    if not df_marketing.empty:
        # Convert date to datetime for merging
        df_profit['Date'] = pd.to_datetime(df_profit['Date'])
        df_marketing['Date'] = pd.to_datetime(df_marketing['Date'])
        
        # Extract year-month for matching
        df_profit['Year_Month'] = df_profit['Date'].dt.to_period('M')
        df_marketing['Year_Month'] = df_marketing['Date'].dt.to_period('M')
        
        # Calculate total sales per category per month
        sales_per_category_month = df_profit.groupby(['Category_Name', 'Year_Month']).size().reset_index(name='Sales_Count')
        
        # Merge marketing costs with sales counts
        marketing_with_counts = df_marketing.merge(
            sales_per_category_month,
            left_on=['Category', 'Year_Month'],
            right_on=['Category_Name', 'Year_Month'],
            how='left'
        )
        
        # Calculate marketing cost per sale
        marketing_with_counts['Marketing_Cost_Per_Sale'] = (
            marketing_with_counts['Marketing_Cost_DZD'] / marketing_with_counts['Sales_Count']
        ).fillna(0)
        
        # Merge back to main dataframe
        df_profit = df_profit.merge(
            marketing_with_counts[['Category', 'Year_Month', 'Marketing_Cost_Per_Sale']],
            left_on=['Category_Name', 'Year_Month'],
            right_on=['Category', 'Year_Month'],
            how='left'
        )
        
        # Fill missing marketing costs with 0
        df_profit['Marketing_Cost_Per_Sale'] = df_profit['Marketing_Cost_Per_Sale'].fillna(0)
        
        print(f"   ✓ Marketing costs allocated (avg per sale: {df_profit['Marketing_Cost_Per_Sale'].mean():.2f} DZD)")
    else:
        print("   ⚠ Marketing data not available, setting marketing cost to 0")
        df_profit['Marketing_Cost_Per_Sale'] = 0
    
    # Step 6: Calculate Net Profit
    print("\n8. Calculating Net Profit...")
    print("   Formula: Net_Profit = Total_Revenue - (Unit_Cost × Quantity) - Shipping_Cost - Marketing_Cost")
    
    df_profit['Product_Cost_Total'] = df_profit['Unit_Cost'] * df_profit['Quantity']
    df_profit['Net_Profit'] = (
        df_profit['Total_Revenue'] - 
        df_profit['Product_Cost_Total'] - 
        df_profit['Shipping_Cost'] - 
        df_profit['Marketing_Cost_Per_Sale']
    )
    
    print(f"   ✓ Net profit calculated for {len(df_profit)} transactions")
    
    # Display statistics
    print("\n9. Net Profit Statistics:")
    print(f"   Total Revenue: {df_profit['Total_Revenue'].sum():,.2f} DZD")
    print(f"   Total Product Costs: {df_profit['Product_Cost_Total'].sum():,.2f} DZD")
    print(f"   Total Shipping Costs: {df_profit['Shipping_Cost'].sum():,.2f} DZD")
    print(f"   Total Marketing Costs: {df_profit['Marketing_Cost_Per_Sale'].sum():,.2f} DZD")
    print(f"   Total Net Profit: {df_profit['Net_Profit'].sum():,.2f} DZD")
    print(f"   Average Net Profit per Transaction: {df_profit['Net_Profit'].mean():.2f} DZD")
    print(f"   Profit Margin: {(df_profit['Net_Profit'].sum() / df_profit['Total_Revenue'].sum() * 100):.2f}%")
    
    # Identify profitable vs unprofitable transactions
    profitable = (df_profit['Net_Profit'] > 0).sum()
    unprofitable = (df_profit['Net_Profit'] <= 0).sum()
    print(f"\n   Profitable transactions: {profitable} ({profitable/len(df_profit)*100:.1f}%)")
    print(f"   Unprofitable transactions: {unprofitable} ({unprofitable/len(df_profit)*100:.1f}%)")
    
    # Select relevant columns for output
    output_columns = [
        'Trans_ID', 'Date', 'Store_ID', 'Product_ID', 'Customer_ID',
        'Quantity', 'Total_Revenue', 'Unit_Cost', 'Product_Cost_Total',
        'Shipping_Cost', 'Marketing_Cost_Per_Sale', 'Net_Profit',
        'Category_Name', 'Region'
    ]
    
    # Filter to only existing columns
    output_columns = [col for col in output_columns if col in df_profit.columns]
    df_profit_output = df_profit[output_columns].copy()
    
    # Save results
    print("\n10. Saving net profit calculations...")
    
    # Create output directory
    profit_dir = "Net Profit Analysis"
    os.makedirs(profit_dir, exist_ok=True)
    
    # Save detailed transaction-level net profit
    profit_detail_path = os.path.join(profit_dir, "transactions_with_net_profit.xlsx")
    df_profit_output.to_excel(profit_detail_path, index=False)
    print(f"   ✓ Saved detailed profit data to: {profit_detail_path}")
    
    # Create summary by product
    product_profit_summary = df_profit.groupby('Product_ID').agg({
        'Total_Revenue': 'sum',
        'Product_Cost_Total': 'sum',
        'Shipping_Cost': 'sum',
        'Marketing_Cost_Per_Sale': 'sum',
        'Net_Profit': 'sum',
        'Quantity': 'sum'
    }).reset_index()
    
    product_profit_summary['Profit_Margin_%'] = (
        product_profit_summary['Net_Profit'] / product_profit_summary['Total_Revenue'] * 100
    ).round(2)
    
    product_profit_path = os.path.join(profit_dir, "product_profit_summary.xlsx")
    product_profit_summary.to_excel(product_profit_path, index=False)
    print(f"   ✓ Saved product profit summary to: {product_profit_path}")
    
    # Create summary by store
    if 'Store_ID' in df_profit.columns:
        store_profit_summary = df_profit.groupby('Store_ID').agg({
            'Total_Revenue': 'sum',
            'Net_Profit': 'sum',
            'Trans_ID': 'count'
        }).reset_index()
        store_profit_summary.columns = ['Store_ID', 'Total_Revenue', 'Net_Profit', 'Transaction_Count']
        store_profit_summary['Profit_Margin_%'] = (
            store_profit_summary['Net_Profit'] / store_profit_summary['Total_Revenue'] * 100
        ).round(2)
        
        store_profit_path = os.path.join(profit_dir, "store_profit_summary.xlsx")
        store_profit_summary.to_excel(store_profit_path, index=False)
        print(f"   ✓ Saved store profit summary to: {store_profit_path}")
    
    # Display sample results
    print("\n11. Sample of transactions with net profit:")
    display_cols = ['Trans_ID', 'Product_ID', 'Quantity', 'Total_Revenue', 'Product_Cost_Total', 'Net_Profit']
    display_cols = [col for col in display_cols if col in df_profit_output.columns]
    print(df_profit_output[display_cols].head(10))
    
    print("\n" + "="*60)
    print("NET PROFIT CALCULATION COMPLETE")
    print("="*60)
    print(f"✓ Calculated net profit for {len(df_profit)} transactions")
    print(f"✓ Total Net Profit: {df_profit['Net_Profit'].sum():,.2f} DZD")
    print(f"✓ Results saved to: {profit_dir}/")


1. Loading cleaned data files...
   ✓ Loaded sales data: 25000 transactions
   ✓ Loaded products data: 38 products
   ✓ Loaded stores data: 12 stores
   ✓ Loaded cities data: 12 cities
   ✓ Loaded category data: 5 categories, 15 subcategories
   ✓ Loaded marketing data: 180 records
   ✓ Loaded shipping rates: 12 records

2. Preparing data for net profit calculation...
   Initial sales records: 25000

3. Merging with product costs...
   ✓ Product costs merged

4. Linking products to categories...
   ✓ Category information added

5. Linking stores to regions for shipping costs...
   ✓ Region information added

6. Calculating shipping costs per transaction...
   ✓ Shipping costs added (avg: 587.22 DZD)

7. Calculating marketing costs per transaction...
   ✓ Marketing costs allocated (avg per sale: 1200.09 DZD)

8. Calculating Net Profit...
   Formula: Net_Profit = Total_Revenue - (Unit_Cost × Quantity) - Shipping_Cost - Marketing_Cost
   ✓ Net profit calculated for 25000 transactions

9.